In [34]:
from sqlalchemy import MetaData
from sqlalchemy import Table, Column, Integer, String, Float, text
from sqlalchemy import create_engine
import pandas as pd

In [97]:
data_path = "/home/hasyim/Bootcamp_AI/capston/Capston3/chatbot/data/raw/imdb_top_1000.csv"
df = pd.read_csv(data_path)

In [98]:
df=df.replace({'Released_Year': 'PG'}, None)

In [99]:
df['Gross'] = df['Gross'].str.replace(',', '', regex=True)

In [100]:
df_clean = df.drop(columns="Overview")

In [101]:
df_clean[['Released_Year','Gross']] = df_clean[['Released_Year','Gross']].apply(pd.to_numeric)

In [102]:
df_clean.index += 1
df_clean.index.name = 'film_id'
df_ke_sql = df_clean.reset_index()

In [103]:
df_ke_sql


,film_id,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,1,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994.0,A,142 min,Drama,9.3,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0
1,2,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972.0,A,175 min,"Crime, Drama",9.2,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411.0
2,3,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008.0,UA,152 min,"Action, Crime, Drama",9.0,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444.0
3,4,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974.0,A,202 min,"Crime, Drama",9.0,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000.0
4,5,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957.0,U,96 min,"Crime, Drama",9.0,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,https://m.media-amazon.com/images/M/MV5BNGEwMT...,Breakfast at Tiffany's,1961.0,A,115 min,"Comedy, Drama, Romance",7.6,76.0,Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen,166544,NaN
996,997,https://m.media-amazon.com/images/M/MV5BODk3Yj...,Giant,1956.0,G,201 min,"Drama, Western",7.6,84.0,George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker,34075,NaN
997,998,https://m.media-amazon.com/images/M/MV5BM2U3Yz...,From Here to Eternity,1953.0,Passed,118 min,"Drama, Romance, War",7.6,85.0,Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed,43374,30500000.0
998,999,https://m.media-amazon.com/images/M/MV5BZTBmMj...,Lifeboat,1944.0,NaN,97 min,"Drama, War",7.6,78.0,Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix,26471,NaN


In [117]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS FILM_TABEL"))

In [118]:
metadata_obj = MetaData()

In [119]:
engine = create_engine('sqlite:////home/hasyim/Bootcamp_AI/capston/Capston3/chatbot/data/process/Coba1-2_capston3.db')

In [120]:
FILM = Table(
    "FILM_TABEL",
    metadata_obj,
    Column("film_id", Integer, primary_key=True),
    Column("Series_Title", String, nullable=False),
    Column("Released_Year", Float),
    Column("Certificate", String),
    Column("Runtime", String),
    Column("Genre", String),
    Column("IMDB_Rating", Float),
    Column("Meta_score", Float),
    Column("Director", String),
    Column("Star1", String),
    Column("Star2", String),
    Column("Star3", String),
    Column("Star4", String),
    Column("No_of_Votes", Integer),
    Column("Gross", Float),
    Column("Poster_Link", String),
    extend_existing=True
)

In [121]:
df_ke_sql.to_sql(
    name='FILM_TABEL',
    con=engine,
    if_exists='append',
    index=False
)

1000

In [122]:
df_new = pd.read_sql('SELECT * FROM FILM_TABEL', con=engine)

In [123]:
print(df_new.head())

   film_id                                        Poster_Link  \
0        1  https://m.media-amazon.com/images/M/MV5BMDFkYT...   
1        2  https://m.media-amazon.com/images/M/MV5BM2MyNj...   
2        3  https://m.media-amazon.com/images/M/MV5BMTMxNT...   
3        4  https://m.media-amazon.com/images/M/MV5BMWMwMG...   
4        5  https://m.media-amazon.com/images/M/MV5BMWU4N2...   

               Series_Title  Released_Year Certificate  Runtime  \
0  The Shawshank Redemption         1994.0           A  142 min   
1             The Godfather         1972.0           A  175 min   
2           The Dark Knight         2008.0          UA  152 min   
3    The Godfather: Part II         1974.0           A  202 min   
4              12 Angry Men         1957.0           U   96 min   

                  Genre  IMDB_Rating  Meta_score              Director  \
0                 Drama          9.3        80.0        Frank Darabont   
1          Crime, Drama          9.2       100.0  Francis 